In [1]:
# https://judge.nitro-ai.org/competitions/slovenska-olympiada/slovak-aoi-2026/1/view

import os 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
from tqdm.auto import tqdm 

In [2]:
train = pd.read_csv("train_data.csv")
test = pd.read_csv("test_data.csv")
subm = pd.read_csv("sample_output.csv")

train.shape, test.shape, subm.shape

((9527, 17), (4084, 17), (4084, 3))

In [3]:
train.head()

,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,58778,968.674,329.370047,227.650499,1.446823,0.722693,59726,273.566215,0.719305,0.984128,0.787172,0.830574,0.005604,0.001645,0.689853,0.998096,BARBUNYA
1,199021,1747.665,705.409896,364.768538,1.933856,0.855924,202063,503.389916,0.783763,0.984945,0.818827,0.713613,0.003544,0.000567,0.509244,0.984805,BOMBAY
2,175717,1585.869,593.695292,380.322431,1.561031,0.767873,178427,473.000881,0.802207,0.984812,0.877989,0.796706,0.003379,0.000840,0.634741,0.990851,BOMBAY
3,38273,704.621,242.712362,200.951906,1.207813,0.560813,38580,220.750305,0.790601,0.992043,0.968704,0.909514,0.006342,0.002677,0.827216,0.999122,SEKER
4,45030,842.585,343.632530,168.215351,2.042813,0.871991,45623,239.445143,0.630761,0.987002,0.797048,0.696806,0.007631,0.001110,0.485538,0.991864,HOROZ


In [4]:
train['Class'].value_counts()

Class
DERMASON    2468
SIRA        1842
SEKER       1442
HOROZ       1373
CALI        1129
BARBUNYA     910
BOMBAY       363
Name: count, dtype: int64

In [5]:
from sklearn.model_selection import train_test_split 
from catboost import Pool

features = [c for c in train.columns if c not in ['Class']]
target_col = 'Class'

X, y = train[features], train[target_col]
X_test = test[features]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)

train_pool = Pool(X_train, y_train)
valid_pool = Pool(X_valid, y_valid)

In [10]:
from catboost import CatBoostClassifier

params = {
    'iterations': 300,
    'loss_function': 'MultiClass',
    'eval_metric': 'TotalF1:average=Weighted',
    'metric_period': 30,
    'learning_rate': 0.1,
    'max_depth': 2,
    'random_state': 42
}

model = CatBoostClassifier(**params)

model.fit(train_pool, eval_set=valid_pool)

0:	learn: 0.5022927	test: 0.4936325	best: 0.4936325 (0)	total: 4.34ms	remaining: 1.3s
30:	learn: 0.8974925	test: 0.8990257	best: 0.8990257 (30)	total: 53.8ms	remaining: 467ms
60:	learn: 0.9127239	test: 0.9161264	best: 0.9161264 (60)	total: 98.4ms	remaining: 386ms
90:	learn: 0.9203173	test: 0.9212151	best: 0.9212151 (90)	total: 145ms	remaining: 332ms
120:	learn: 0.9240786	test: 0.9265129	best: 0.9265129 (120)	total: 186ms	remaining: 275ms
150:	learn: 0.9283451	test: 0.9297464	best: 0.9297464 (150)	total: 232ms	remaining: 229ms
180:	learn: 0.9294915	test: 0.9307620	best: 0.9307620 (180)	total: 277ms	remaining: 182ms
210:	learn: 0.9314371	test: 0.9328361	best: 0.9328361 (210)	total: 319ms	remaining: 135ms
240:	learn: 0.9328677	test: 0.9328794	best: 0.9328794 (240)	total: 363ms	remaining: 88.8ms
270:	learn: 0.9334318	test: 0.9349548	best: 0.9349548 (270)	total: 409ms	remaining: 43.8ms
299:	learn: 0.9352962	test: 0.9370540	best: 0.9370540 (299)	total: 455ms	remaining: 0us

bestTest = 0.9370

In [14]:
preds = model.predict(X_test).flatten()
subm['answer'] = preds
subm.to_csv("submission.csv", index=False)
subm.head()

,subtaskID,datapointID,answer
0,1,0,CALI
1,1,1,SEKER
2,1,2,SEKER
3,1,3,DERMASON
4,1,4,CALI


In [15]:
subm['answer'].value_counts()

answer
DERMASON    1109
SIRA         778
SEKER        584
HOROZ        545
CALI         515
BARBUNYA     393
BOMBAY       160
Name: count, dtype: int64